In [8]:
import os
import math
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

import matplotlib.pyplot as plt

from tslearn.clustering import TimeSeriesKMeans

In [9]:
path = './data/nifty500_5yr_daily/'
ex_rate_path = './data/usd_to_inr/'
symbol_path = './data/ticker/'
price_col = 'avg' # CH_TRADE_AVG_PRICE
date_col = 'date' # CH_TIMESTAMP

In [10]:
# reusable function to fill saturday and sunday with preceding non null values (ffill method)
# used for filling both stock price and exchange rate
def fill_blanks(df:pd.DataFrame, date_col:'str', price_col:'str') -> pd.DataFrame:
    df_date = pd.DataFrame(
        {
            date_col : pd.date_range(df[date_col].min(), df[date_col].max()).astype(str)
        }
    )
    df_expanded = df_date.merge(df, on = date_col, how='left').sort_values(date_col)
    df_expanded[price_col] = df_expanded[price_col].ffill()
    return df_expanded

In [11]:
df_ticker = pd.read_csv('./data/ticker/screener_to_ticker.csv')
df_screener = pd.read_csv('./data/screener_profitable_list.csv')
df_named = df_screener.merge(df_ticker, on='Short_Name')[['Short_Name', 'Ticker']]
df_named.to_csv('./data/shortlist.csv', index=None)

In [12]:
ex_rate_files = sorted(set(os.listdir(ex_rate_path)))
ex_rate_df_list = []
for file in ex_rate_files:
    df = pd.read_csv(ex_rate_path + file)
    ex_rate_df_list.append(df)

df_ex_rate = pd.concat(ex_rate_df_list)
df_ex_rate['USD_to_INR'] = df_ex_rate.apply(lambda row : (row['Open'] + row['High'] + row['Low'] + row['Close'])/4, axis=1)
df_ex_rate['Date'] = pd.to_datetime(df_ex_rate['Date']).astype(str)
df_ex_rate.drop(columns=['Open', 'High', 'Low', 'Close'], inplace=True)
df_ex_rate = fill_blanks(df_ex_rate, 'Date', 'USD_to_INR')

In [15]:
# files = sorted(set(os.listdir(path)))
# files = [file for file in files if file.endswith('csv')]
symbols = df_named.loc[df_named['Ticker'].str.contains('ADANI|YESBANK')==False]['Ticker']

mySeries = []
namesofMySeries = []
named_df_dict = {}
ads_for_change_det = []

for symbol in symbols :
    # read historic price data
    file = symbol.replace('-', '_') + '.csv'
    try :
        df = pd.read_csv(path+file)[[date_col, price_col]]
    except FileNotFoundError :
        print(f'{symbol} file not found')
        continue
    # drop if there are duplicates
    df.drop_duplicates(date_col, inplace=True)
    df = fill_blanks(df, date_col, price_col)
    # map inr to usd conversion based on date
    df = df.merge(df_ex_rate, left_on=date_col, right_on='Date', how='left')
    # convert inr to usd
    df['price_in_usd'] = df[price_col] / df['USD_to_INR']
    # drop other columns
    df.drop(columns=['Date', price_col, 'USD_to_INR'], inplace=True)
    
    df['date'] = pd.to_datetime(df[date_col])
    df['year'] = df['date'].dt.isocalendar().year.astype(str)
    df['week'] = df['date'].dt.isocalendar().week.astype(str).str.zfill(2)
    df['year_week'] = df['year'] + df['week']

    df_7d_avg = df.groupby('year_week', as_index=False).agg(price_in_usd_7d_avg = ('price_in_usd', 'mean'))
    # index the first day value to 100 units for easier understanding of growth scale
    factor_rs100 = df_7d_avg[['price_in_usd_7d_avg']].iloc[0,0] / 100 # converting to "Rs. 100 invested in ..." format
    df_7d_avg['price_in_usd_7d_avg_ind'] = (df_7d_avg['price_in_usd_7d_avg'] / factor_rs100).round().astype(int)
    ads = df_7d_avg.copy()
    ads['symbol'] = symbol
    ads_for_change_det.append(ads)
    df_7d_avg.drop(columns=['year_week', 'price_in_usd_7d_avg'], inplace=True)
    # df_7d_avg = df_7d_avg.tail(156) # last 3 years
    if len(df_7d_avg) == 223 : # 156 for 3 years and 217 for whole data
        mySeries.append(df_7d_avg)
        namesofMySeries.append(symbol)
        named_df_dict[symbol] = df_7d_avg
    else :
        print(f'{symbol} omitted')
pd.concat(ads_for_change_det).to_csv('./data/ads_for_change_det.csv', index=None)

LICI omitted
AIIL omitted
EMBASSY file not found
ISEC omitted
BAJAJHFL omitted
MANKIND omitted
JIOFIN omitted
IREDA omitted
LLOYDSME omitted
JSWINFRA omitted
AWL omitted
BHARTIHEXA omitted
FIVESTAR omitted
543225 file not found
ABSLAMC omitted
NUVAMA omitted
AADHARHFC omitted
 VTL file not found
STARHEALTH omitted
APTUS omitted
ZOMATO omitted
MSUMI omitted
EMCURE omitted
MINDSPACE file not found
NXST file not found
MEDANTA omitted
SPLPETRO omitted
METROBRAND omitted
MANYAVAR omitted
INDGN omitted
GODIGIT omitted
CELLO omitted
CONCORDBIO omitted
KFINTECH omitted
JYOTICNC omitted
ANANDRATHI omitted
BIKAJI omitted
CLEAN omitted
KAYNES omitted
NHIT omitted
RAINBOW omitted
BLUEJET omitted
TBOTEK omitted
POLICYBZR omitted
DOMS omitted
509046 file not found
VIJAYA omitted
AETHER omitted
IWEL file not found
BIRET file not found
SIGNATURE omitted
NYKAA omitted
PTCIL omitted


In [ ]:
n_rows = int(np.ceil(len(mySeries)/4))

fig, axs = plt.subplots(n_rows,4,figsize=(n_rows/8,n_rows))
fig.suptitle('Series')
for i in range(n_rows):
    for j in range(4):
        if i*4+j+1>len(mySeries): # pass the others that we can't fill
            continue
        axs[i, j].plot(mySeries[i*4+j].values)
        axs[i, j].set_title(namesofMySeries[i*4+j])
fig.tight_layout()
# plt.savefig('nifty_500_5yr_30D_mov_avg_infl_adj_usd.png')
plt.show()

In [ ]:
som_x = som_y = math.ceil(math.sqrt(math.sqrt(len(mySeries))))

In [ ]:
cluster_count = 169 # arbitary perfect square to keep plotting simple
# yet to figure out a good method

km = TimeSeriesKMeans(n_clusters=cluster_count, metric="dtw")

labels = km.fit_predict(mySeries)

In [ ]:
pd.DataFrame(labels,columns=['cluster_labels']).to_csv('./data/cluster_labels.csv', index=None)

In [ ]:
df_clusters = pd.DataFrame({'Symbol':namesofMySeries, 'Cluster_ID' : labels})
df_clust_count = (
    df_clusters
    .groupby('Cluster_ID', as_index=False)
    .agg(
        Member_Count = ('Symbol', 'nunique'),
        Member_List = ('Symbol', lambda x : list(x))
    )
)
df_final = df_clust_count.loc[df_clust_count['Member_Count'] > 1].sort_values('Member_Count', ascending=False).reset_index(drop=True)
df_final

In [ ]:
plot_count = int(np.ceil(np.sqrt(len(df_final))))

fig, axs = plt.subplots(plot_count,plot_count,figsize=(40,40))
fig.suptitle('Clusters')
row_i=0
column_j=0
# For each label there is,
# plots every series with that label
for cluster in df_final['Member_List'] :
    symbols = ','.join(cluster)
    for member in cluster :
        df = named_df_dict[member]
        axs[row_i, column_j].plot(df,c="blue",alpha=0.4)
    axs[row_i, column_j].set_title(symbols)
    column_j+=1
    if column_j%plot_count == 0:
        row_i+=1
        column_j=0
fig.tight_layout()